# Capstone, a context assembler

**Scenario:** a tutoring service plans weekly practice for each learner. The planner reads the rules,
the learner record, a summary of earlier sessions and the last few messages. One week it named a
learner id that does not exist.

Nothing was missing from the system. The assembler ran out of room before it reached the part that
mattered.

This capstone joins the two lessons before it. Scoped rules and pinned facts only help if something
decides the order they go in. Think of it as packing a bag to an airline weight limit. The order you
pack in decides what gets left on the bed.

## Mechanics

Four blocks, one budget, and a fixed order.

| Block | Where it comes from | Position | Can it be dropped |
|---|---|---|---|
| Activated rules | the path resolver | first, and stable across turns | no |
| Pinned facts | lifted out before any summary | second | no |
| Summary of older sessions | a small model | third | yes, first to go |
| Recent turns | the live conversation, word for word | last, nearest the question | trimmed oldest first |

The order is not decoration. Rules and facts sit at the front because they change least, and the live
turns sit at the end because they matter most to the next answer.

## The picture

![Four blocks assembled in a fixed order under one budget](images/context-assembler.svg)

One function owns the whole request. If two places can append to a prompt, the budget is a wish.

## The cost

```
assembled = rules + facts + summary + recent turns
gate      : assembled <= budget, and budget is far below the model's context window
```

The budget is yours to choose, not the provider's to give. The context window is what the model can
hold. The budget is what you will pay for on every turn, forever.

## The failure

The four blocks, as a tutoring service would actually hold them.

In [1]:
LEARNER = "LR-2024-0881"
RULES = ("## Assessment rules\n"
         "A learner with an extra time accommodation is never given a timed drill.\n"
         "Name the learner id exactly as it appears in the record.")
FACTS = (f"## Learner record\nlearner_id: {LEARNER}\n"
         "accommodation: extra time on all assessments, no timed drills\n"
         "target: pass the Year 9 algebra retake in May")
SUMMARY = ("## Earlier sessions\nSteady on linear equations, weak on factorising quadratics, "
           "loses marks on sign errors. Attendance has been good all term.")
RECENT = ["Tutor: last week we worked through factorising by grouping.",
          "Learner: I get the grouping part but I keep flipping the signs.",
          "Tutor: sign errors again on question 4 and question 7."]
OLDER = [f"Session note {i}: practice set {i} done, mixed accuracy, homework in on time."
         for i in range(40)]
ASK = "Plan this week's practice. Name the learner id and say what to avoid."

One call, and it reports what the provider charged. Every final number below comes from that field.

In [2]:
from vault import get_client, load_env, model_for, provider_truth

load_env()
client = get_client("04-context-engineering/03-capstone-a-context-assembler")


def ask_plan(context):
    """One planning turn. Returns the answer and the prompt tokens charged for it."""
    reply = client.chat.completions.create(
        model=model_for("default"), max_tokens=250,
        messages=[{"role": "system", "content": context},
                  {"role": "user", "content": ASK}])
    return reply.choices[0].message.content, reply.usage.prompt_tokens

An assembler tries many arrangements per turn, so it cannot ask the provider what each one costs. It
needs a cheap local estimate. Calibrate one against a real charge, once.

In [3]:
CORPUS = "\n".join([RULES, FACTS, SUMMARY] + RECENT + OLDER)
_, charged_for_corpus = ask_plan(CORPUS)
CHARS_PER_TOKEN = len(CORPUS) / charged_for_corpus
WINDOW = provider_truth()["models"][model_for("default")]["context_length"]

print(f"{len(CORPUS)} characters were charged as {charged_for_corpus} prompt tokens")
print(f"so about {CHARS_PER_TOKEN:.2f} characters per token for this text")
print(f"the model's context window is {WINDOW} tokens, and the budget below is far under it")

3642 characters were charged as 1030 prompt tokens
so about 3.54 characters per token for this text
the model's context window is 1048576 tokens, and the budget below is far under it


That ratio is an approximation and this notebook will not pretend otherwise. It counts the question
and the message framing as corpus, so it runs generous. Good enough to choose an arrangement, and
every final number below is the charge, not the estimate.

In [4]:
def approx_tokens(text):
    """A character based estimate, on purpose. The provider's count is the truth."""
    return round(len(text) / CHARS_PER_TOKEN)

Now the assembler most services write first. It fills from the newest message backwards until the
budget runs out, because recent context feels like the valuable kind.

In [5]:
def assemble_naive(budget):
    """Newest first, until the budget is gone. Rules and facts are last in line."""
    kept, used = [], 0
    for block in RECENT[::-1] + OLDER[::-1] + [SUMMARY, FACTS, RULES]:
        cost = approx_tokens(block)
        if used + cost > budget:
            continue
        kept.append(block)
        used += cost
    return "\n".join(kept), used

Give it a budget a service would run at, and ask for the plan.

In [6]:
BUDGET = 400
naive_context, naive_used = assemble_naive(BUDGET)
naive_answer, naive_charged = ask_plan(naive_context)

print(f"estimate {naive_used} tokens, charged {naive_charged}, budget {BUDGET}")
print(f"rules survived the pack : {'Assessment rules' in naive_context}")
print(f"learner record survived : {LEARNER in naive_context}")
print(f"\nplan says: {naive_answer.strip()[:200]}")
assert LEARNER in naive_answer, "the plan did not name the learner id from the record"

estimate 386 tokens, charged 414, budget 400
rules survived the pack : False
learner record survived : False

plan says: **Learner ID:** L-39

**This week's practice plan:**

This week, we will continue to focus on **factorising by grouping**. We will build on the skills you've developed so far, with a particular emphas


AssertionError: the plan did not name the learner id from the record

## The diagnosis

The assertion fires. The plan names a learner id that was never in the request, invented from a
session note, and says nothing about the extra time the record requires.

Read the mechanics table against what the pack did. Rules and facts were last in line, so forty
session notes with no bearing on this week filled the bag first. Both blocks that cannot be dropped
were the blocks that got dropped.

The estimate also came in under the budget while the charge came in over it, which is what a
character based count does. Asked to name a learner id with none in front of it, the model produced
one that looked right, exactly as it did with the missing lot id before this.

## The fix

One function owns the order, and the two blocks that must never be dropped go in before anything else
competes for room. If those alone do not fit, that is a configuration error and it says so at once.

In [7]:
def assemble(rules, facts, summary, recent, budget):
    """Rules, facts, summary, recent turns. The summary goes first when room runs out."""
    head, used = [rules, facts], approx_tokens(rules) + approx_tokens(facts)
    if used > budget:
        raise ValueError(f"rules and facts need {used} tokens, budget is {budget}")
    tail = []
    for turn in list(recent)[::-1]:
        if used + approx_tokens(turn) > budget:
            break
        tail.insert(0, turn)
        used += approx_tokens(turn)
    if used + approx_tokens(summary) <= budget:
        head.append(summary)
        used += approx_tokens(summary)
    return "\n".join(head + tail), used

Recent turns are taken newest first and written back in their original order, because a conversation
read backwards is a different conversation. Same budget, same blocks, same question.

In [8]:
packed, used = assemble(RULES, FACTS, SUMMARY, RECENT, BUDGET)
answer, charged = ask_plan(packed)

print(f"before: {naive_charged} charged, learner id named: {LEARNER in naive_answer}, "
      f"extra time named: {'extra time' in naive_answer.lower()}")
print(f"after : {charged} charged, learner id named: {LEARNER in answer}, "
      f"extra time named: {'extra time' in answer.lower()}")
print(f"        estimate said {used}, budget was {BUDGET}")
print(f"\nplan says: {answer.strip()[:220]}")

before: 414 charged, learner id named: False, extra time named: False
after : 170 charged, learner id named: True, extra time named: True
        estimate said 174, budget was 400

plan says: This week's practice for learner LR-2024-0881 will focus on:

*   **Continued practice with factorising quadratics**, with a specific emphasis on **identifying and correcting sign errors**.
*   **Review of linear equatio


## The gate

Two properties, one check, no model call. The context stays inside the budget, and the blocks that
cannot be dropped are still there when the budget is tight.

In [9]:
def test_the_budget_holds_and_the_record_survives():
    for budget in (120, 400, 2000):
        packed, used = assemble(RULES, FACTS, SUMMARY, RECENT * 30, budget)
        assert used <= budget, f"assembled {used} tokens against a budget of {budget}"
        assert LEARNER in packed, f"the learner record was evicted at a budget of {budget}"
    try:
        assemble(RULES, FACTS, SUMMARY, RECENT, 10)
    except ValueError:
        return
    raise AssertionError("a budget too small for the rules did not raise")


test_the_budget_holds_and_the_record_survives()
print("gate holds: the budget is never exceeded and the record is never evicted")

gate holds: the budget is never exceeded and the record is never evicted


Move `facts` into the loop that trims turns and this test fails at the tightest budget.

### Enterprise exploration

- The estimate runs generous, so a real charge can land above the budget. What does that cost at a
  million turns a day, and where would you put the hard stop?
- Two teams both want to append to the prompt. What stops the second one, and how would a review
  catch a bypass before it ships?
- A tutor disputes a plan from last term. Which four blocks were in that request, and where is that
  recorded for an audit?
- Dropping the summary keeps the budget and loses the term's progress. Who decides that trade off,
  and how does a learner find out?

### Key takeaways

- Order is a decision. Filling newest first drops the blocks that cannot be dropped.
- Rules and facts at the front, the summary in the middle, the live turns at the end.
- Estimate locally to choose an arrangement, then report the charge the provider made.
- A budget with no check is a wish. Assert it, including the case where the fixed blocks do not fit.